In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, SGDRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.ensemble import StackingRegressor, VotingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import warnings
warnings.filterwarnings('ignore')

# Load all datasets
print("=" * 80)
print("LOADING DATASETS")
print("=" * 80)

booknow_booking = pd.read_csv("./Cinema_Audience_Forecasting_challenge/booknow_booking/booknow_booking.csv")
# booknow_theaters = pd.read_csv('/kaggle/input/Cinema_Audience_Forecasting_challenge/booknow_theaters/booknow_theaters.csv')
# booknow_visits = pd.read_csv('/kaggle/input/Cinema_Audience_Forecasting_challenge/booknow_visits/booknow_visits.csv')
# cinePOS_booking = pd.read_csv('/kaggle/input/Cinema_Audience_Forecasting_challenge/cinePOS_booking/cinePOS_booking.csv')
# cinePOS_theaters = pd.read_csv('/kaggle/input/Cinema_Audience_Forecasting_challenge/cinePOS_theaters/cinePOS_theaters.csv')
# date_info = pd.read_csv('/kaggle/input/Cinema_Audience_Forecasting_challenge/date_info/date_info.csv')
# movie_theater_id_relation = pd.read_csv('/kaggle/input/Cinema_Audience_Forecasting_challenge/movie_theater_id_relation/movie_theater_id_relation.csv')
# sample_submission = pd.read_csv('/kaggle/input/Cinema_Audience_Forecasting_challenge/sample_submission/sample_submission.csv')

# print(f"Booknow Booking: {booknow_booking.shape}")
# print(f"Booknow Theaters: {booknow_theaters.shape}")
# print(f"Booknow Visits: {booknow_visits.shape}")
# print(f"CinePOS Booking: {cinePOS_booking.shape}")
# print(f"CinePOS Theaters: {cinePOS_theaters.shape}")
# print(f"Date Info: {date_info.shape}")
# print(f"Theater ID Relation: {movie_theater_id_relation.shape}")
# print(f"Sample Submission: {sample_submission.shape}")

# # ============================================================================
# # EXPLORATORY DATA ANALYSIS
# # ============================================================================
# print("\n" + "=" * 80)
# print("EXPLORATORY DATA ANALYSIS")
# print("=" * 80)

# # Convert datetime columns
# booknow_booking['show_datetime'] = pd.to_datetime(booknow_booking['show_datetime'])
# booknow_booking['booking_datetime'] = pd.to_datetime(booknow_booking['booking_datetime'])
# booknow_visits['show_date'] = pd.to_datetime(booknow_visits['show_date'])
# date_info['show_date'] = pd.to_datetime(date_info['show_date'])

# cinePOS_booking['show_datetime'] = pd.to_datetime(cinePOS_booking['show_datetime'])
# cinePOS_booking['booking_datetime'] = pd.to_datetime(cinePOS_booking['booking_datetime'])

# print("\nBooknow Visits Statistics:")
# print(booknow_visits['audience_count'].describe())

# print("\nMissing Values in Booknow Theaters:")
# print(booknow_theaters.isnull().sum())

# print("\nTheater Types Distribution:")
# print(booknow_theaters['theater_type'].value_counts())

# # Parse submission IDs to understand prediction requirements
# sample_submission['theater_id'] = sample_submission['ID'].str.split('_').str[0] + '_' + sample_submission['ID'].str.split('_').str[1]
# sample_submission['pred_date'] = pd.to_datetime(sample_submission['ID'].str.split('_').str[2])

# print(f"\nPrediction Date Range: {sample_submission['pred_date'].min()} to {sample_submission['pred_date'].max()}")
# print(f"Unique Theaters to Predict: {sample_submission['theater_id'].nunique()}")

# # ============================================================================
# # FEATURE ENGINEERING
# # ============================================================================
# print("\n" + "=" * 80)
# print("FEATURE ENGINEERING")
# print("=" * 80)

# # Merge booknow_visits with date_info
# data = booknow_visits.merge(date_info, on='show_date', how='left')

# # Add theater information
# data = data.merge(booknow_theaters, on='book_theater_id', how='left')

# # Extract temporal features
# data['year'] = data['show_date'].dt.year
# data['month'] = data['show_date'].dt.month
# data['day'] = data['show_date'].dt.day
# data['day_of_year'] = data['show_date'].dt.dayofyear
# data['week'] = data['show_date'].dt.isocalendar().week
# data['is_weekend'] = data['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)

# # Create aggregated features from booking data
# booknow_booking['show_date'] = booknow_booking['show_datetime'].dt.date
# booknow_booking['show_date'] = pd.to_datetime(booknow_booking['show_date'])

# # Aggregate bookings by theater and date
# booking_agg = booknow_booking.groupby(['book_theater_id', 'show_date']).agg({
#     'tickets_booked': ['sum', 'mean', 'count', 'std']
# }).reset_index()
# booking_agg.columns = ['book_theater_id', 'show_date', 'total_tickets', 'avg_tickets', 'num_bookings', 'std_tickets']
# booking_agg['std_tickets'].fillna(0, inplace=True)

# # Merge booking features
# data = data.merge(booking_agg, left_on=['book_theater_id', 'show_date'], 
#                   right_on=['book_theater_id', 'show_date'], how='left')

# # Fill missing booking features with 0
# data[['total_tickets', 'avg_tickets', 'num_bookings', 'std_tickets']] = \
#     data[['total_tickets', 'avg_tickets', 'num_bookings', 'std_tickets']].fillna(0)

# # Historical features (rolling averages)
# data = data.sort_values(['book_theater_id', 'show_date'])
# data['audience_lag_1'] = data.groupby('book_theater_id')['audience_count'].shift(1)
# data['audience_lag_7'] = data.groupby('book_theater_id')['audience_count'].shift(7)
# data['audience_rolling_7'] = data.groupby('book_theater_id')['audience_count'].transform(
#     lambda x: x.rolling(7, min_periods=1).mean())
# data['audience_rolling_30'] = data.groupby('book_theater_id')['audience_count'].transform(
#     lambda x: x.rolling(30, min_periods=1).mean())

# # Theater-level statistics
# theater_stats = data.groupby('book_theater_id')['audience_count'].agg(['mean', 'std', 'min', 'max']).reset_index()
# theater_stats.columns = ['book_theater_id', 'theater_avg', 'theater_std', 'theater_min', 'theater_max']
# data = data.merge(theater_stats, on='book_theater_id', how='left')

# # Day of week statistics per theater
# dow_stats = data.groupby(['book_theater_id', 'day_of_week'])['audience_count'].mean().reset_index()
# dow_stats.columns = ['book_theater_id', 'day_of_week', 'dow_avg']
# data = data.merge(dow_stats, on=['book_theater_id', 'day_of_week'], how='left')

# # Encode categorical variables
# le_theater = LabelEncoder()
# le_area = LabelEncoder()
# le_type = LabelEncoder()
# le_dow = LabelEncoder()

# data['theater_id_encoded'] = le_theater.fit_transform(data['book_theater_id'].astype(str))
# data['theater_area_encoded'] = le_area.fit_transform(data['theater_area'].fillna('Unknown'))
# data['theater_type_encoded'] = le_type.fit_transform(data['theater_type'].fillna('Unknown'))
# data['day_of_week_encoded'] = le_dow.fit_transform(data['day_of_week'])

# print(f"Total features created: {data.shape[1]}")
# print(f"Data shape: {data.shape}")

# # ============================================================================
# # DATA VISUALIZATION
# # ============================================================================
# print("\n" + "=" * 80)
# print("DATA VISUALIZATION")
# print("=" * 80)

# fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# # Audience distribution
# axes[0, 0].hist(data['audience_count'], bins=50, edgecolor='black')
# axes[0, 0].set_title('Audience Count Distribution')
# axes[0, 0].set_xlabel('Audience Count')
# axes[0, 0].set_ylabel('Frequency')

# # Day of week
# dow_avg = data.groupby('day_of_week')['audience_count'].mean().reindex(
#     ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'])
# axes[0, 1].bar(range(7), dow_avg.values)
# axes[0, 1].set_xticks(range(7))
# axes[0, 1].set_xticklabels(dow_avg.index, rotation=45)
# axes[0, 1].set_title('Average Audience by Day of Week')
# axes[0, 1].set_ylabel('Average Audience')

# # Monthly trend
# monthly = data.groupby('month')['audience_count'].mean()
# axes[0, 2].plot(monthly.index, monthly.values, marker='o')
# axes[0, 2].set_title('Average Audience by Month')
# axes[0, 2].set_xlabel('Month')
# axes[0, 2].set_ylabel('Average Audience')

# # Theater type
# theater_type_avg = data.groupby('theater_type')['audience_count'].mean()
# axes[1, 0].bar(range(len(theater_type_avg)), theater_type_avg.values)
# axes[1, 0].set_xticks(range(len(theater_type_avg)))
# axes[1, 0].set_xticklabels(theater_type_avg.index, rotation=45)
# axes[1, 0].set_title('Average Audience by Theater Type')

# # Weekend vs Weekday
# weekend_data = data.groupby('is_weekend')['audience_count'].mean()
# axes[1, 1].bar(['Weekday', 'Weekend'], weekend_data.values)
# axes[1, 1].set_title('Weekday vs Weekend Audience')
# axes[1, 1].set_ylabel('Average Audience')

# # Correlation heatmap (top features)
# numeric_cols = ['audience_count', 'total_tickets', 'avg_tickets', 'num_bookings', 
#                 'audience_lag_1', 'audience_rolling_7', 'is_weekend', 'month', 'week']
# corr_data = data[numeric_cols].dropna()
# corr = corr_data.corr()
# sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1, 2], cbar_kws={'shrink': 0.8})
# axes[1, 2].set_title('Feature Correlation Matrix')

# plt.tight_layout()
# plt.savefig('eda_visualization.png', dpi=100, bbox_inches='tight')
# print("Visualizations saved to 'eda_visualization.png'")

# # ============================================================================
# # PREPARE TRAINING DATA
# # ============================================================================
# print("\n" + "=" * 80)
# print("PREPARING TRAINING DATA")
# print("=" * 80)

# # Select features for modeling
# feature_cols = [
#     'theater_id_encoded', 'theater_area_encoded', 'theater_type_encoded',
#     'day_of_week_encoded', 'month', 'day', 'week', 'day_of_year',
#     'is_weekend', 'total_tickets', 'avg_tickets', 'num_bookings', 'std_tickets',
#     'audience_lag_1', 'audience_lag_7', 'audience_rolling_7', 'audience_rolling_30',
#     'theater_avg', 'theater_std', 'dow_avg', 'latitude', 'longitude'
# ]

# # Remove rows with NaN in key features
# train_data = data[feature_cols + ['audience_count']].dropna()

# X = train_data[feature_cols]
# y = train_data['audience_count']

# print(f"Training data shape: X={X.shape}, y={y.shape}")

# # Train-test split
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Scale features
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

# print(f"Train set: {X_train.shape}, Test set: {X_test.shape}")

# # ============================================================================
# # MODEL BUILDING - 7 DIFFERENT MODELS
# # ============================================================================
# print("\n" + "=" * 80)
# print("BUILDING 7 DIFFERENT MODELS")
# print("=" * 80)

# models = {
#     '1. Linear Regression': LinearRegression(),
#     '2. Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
#     '3. Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
#     '4. XGBoost': XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1),
#     '5. LightGBM': LGBMRegressor(n_estimators=100, random_state=42, n_jobs=-1, verbose=-1),
#     '6. Support Vector Regression': SVR(kernel='rbf', C=1.0),
#     '7. Neural Network (MLP)': MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42)
# }

# results = []

# for name, model in models.items():
#     print(f"\nTraining {name}...")
    
#     # Use scaled data for models that benefit from it
#     if name in ['6. Support Vector Regression', '7. Neural Network (MLP)']:
#         X_tr, X_te = X_train_scaled, X_test_scaled
#     else:
#         X_tr, X_te = X_train, X_test
    
#     # Train model
#     model.fit(X_tr, y_train)
    
#     # Predictions
#     y_pred_train = model.predict(X_tr)
#     y_pred_test = model.predict(X_te)
    
#     # Ensure non-negative predictions
#     y_pred_train = np.maximum(y_pred_train, 0)
#     y_pred_test = np.maximum(y_pred_test, 0)
    
#     # Evaluation metrics
#     train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
#     test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
#     train_r2 = r2_score(y_train, y_pred_train)
#     test_r2 = r2_score(y_test, y_pred_test)
#     test_mae = mean_absolute_error(y_test, y_pred_test)
    
#     results.append({
#         'Model': name,
#         'Train RMSE': train_rmse,
#         'Test RMSE': test_rmse,
#         'Train R2': train_r2,
#         'Test R2': test_r2,
#         'Test MAE': test_mae
#     })
    
#     print(f"  Train RMSE: {train_rmse:.4f}, Train R2: {train_r2:.4f}")
#     print(f"  Test RMSE: {test_rmse:.4f}, Test R2: {test_r2:.4f}, Test MAE: {test_mae:.4f}")

# # Display results
# results_df = pd.DataFrame(results)
# results_df = results_df.sort_values('Test R2', ascending=False)
# print("\n" + "=" * 80)
# print("MODEL COMPARISON")
# print("=" * 80)
# print(results_df.to_string(index=False))

# # ============================================================================
# # SELECT BEST MODEL FOR PREDICTIONS
# # ============================================================================
# print("\n" + "=" * 80)
# print("SELECTING BEST MODEL")
# print("=" * 80)

# # Get the best model based on Test R2 score
# best_model_name = results_df.iloc[0]['Model']
# print(f"Best Model: {best_model_name}")
# print(f"Test R2: {results_df.iloc[0]['Test R2']:.4f}")
# print(f"Test RMSE: {results_df.iloc[0]['Test RMSE']:.4f}")

# # Store the best model
# best_model_dict = dict(zip(models.keys(), models.values()))
# best_model = best_model_dict[best_model_name]

# # Re-train on full training data
# print("\nRe-training best model on full training data...")
# if best_model_name in ['6. Support Vector Regression', '7. Neural Network (MLP)']:
#     best_model.fit(X_train_scaled, y_train)
# else:
#     best_model.fit(X_train, y_train)

# # ============================================================================
# # GENERATE PREDICTIONS FOR SUBMISSION
# # ============================================================================
# print("\n" + "=" * 80)
# print("GENERATING FINAL PREDICTIONS")
# print("=" * 80)



LOADING DATASETS


FileNotFoundError: [Errno 2] No such file or directory: './Cinema_Audience_Forecasting_challenge/booknow_booking/booknow_booking.csv'

In [ ]:
# GENERATE PREDICTIONS FOR SUBMISSION
# ============================================================================
print("\n" + "=" * 80)
print("GENERATING FINAL PREDICTIONS")
print("=" * 80)

# best_model is already defined in the previous section
print(f"Using {best_model_name} for final predictions")

# Prepare test data for submission
sample_submission['year'] = sample_submission['pred_date'].dt.year
sample_submission['month'] = sample_submission['pred_date'].dt.month
sample_submission['day'] = sample_submission['pred_date'].dt.day
sample_submission['day_of_year'] = sample_submission['pred_date'].dt.dayofyear
sample_submission['week'] = sample_submission['pred_date'].dt.isocalendar().week

# Merge with date_info
sample_submission = sample_submission.merge(date_info, left_on='pred_date', right_on='show_date', how='left')
sample_submission['is_weekend'] = sample_submission['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)

# Prepare prediction features
test_features = []
for idx, row in sample_submission.iterrows():
    theater_id = row['theater_id']
    
    # Get theater-specific historical data
    theater_data = data[data['book_theater_id'] == theater_id]
    
    if len(theater_data) > 0:
        theater_encoded = theater_data['theater_id_encoded'].iloc[0]
        area_encoded = theater_data['theater_area_encoded'].iloc[0]
        type_encoded = theater_data['theater_type_encoded'].iloc[0]
        theater_avg_val = theater_data['theater_avg'].iloc[0]
        theater_std_val = theater_data['theater_std'].iloc[0]
        lat = theater_data['latitude'].iloc[0]
        lon = theater_data['longitude'].iloc[0]
        
        # Get day of week average
        dow_data = theater_data[theater_data['day_of_week'] == row['day_of_week']]
        dow_avg_val = dow_data['dow_avg'].iloc[0] if len(dow_data) > 0 else theater_avg_val
        
        # Get recent historical values
        recent_data = theater_data.tail(30)
        audience_lag_1_val = recent_data['audience_count'].iloc[-1] if len(recent_data) > 0 else theater_avg_val
        audience_lag_7_val = recent_data['audience_count'].iloc[-7] if len(recent_data) >= 7 else theater_avg_val
        audience_rolling_7_val = recent_data['audience_count'].tail(7).mean()
        audience_rolling_30_val = recent_data['audience_count'].mean()
        
        # Estimate booking features (use historical average)
        total_tickets_val = theater_data['total_tickets'].mean()
        avg_tickets_val = theater_data['avg_tickets'].mean()
        num_bookings_val = theater_data['num_bookings'].mean()
        std_tickets_val = theater_data['std_tickets'].mean()
    else:
        # Use global averages for unknown theaters
        theater_encoded = 0
        area_encoded = 0
        type_encoded = 0
        theater_avg_val = data['audience_count'].mean()
        theater_std_val = data['audience_count'].std()
        lat = data['latitude'].mean()
        lon = data['longitude'].mean()
        dow_avg_val = theater_avg_val
        audience_lag_1_val = theater_avg_val
        audience_lag_7_val = theater_avg_val
        audience_rolling_7_val = theater_avg_val
        audience_rolling_30_val = theater_avg_val
        total_tickets_val = data['total_tickets'].mean()
        avg_tickets_val = data['avg_tickets'].mean()
        num_bookings_val = data['num_bookings'].mean()
        std_tickets_val = data['std_tickets'].mean()
    
    # Encode day of week
    try:
        dow_encoded = le_dow.transform([row['day_of_week']])[0]
    except:
        dow_encoded = 0
    
    test_features.append([
        theater_encoded, area_encoded, type_encoded, dow_encoded,
        row['month'], row['day'], row['week'], row['day_of_year'],
        row['is_weekend'], total_tickets_val, avg_tickets_val, num_bookings_val, std_tickets_val,
        audience_lag_1_val, audience_lag_7_val, audience_rolling_7_val, audience_rolling_30_val,
        theater_avg_val, theater_std_val, dow_avg_val, lat, lon
    ])

X_submission = pd.DataFrame(test_features, columns=feature_cols)

# Make predictions
predictions = best_model.predict(X_submission)
predictions = np.maximum(predictions, 0)  # Ensure non-negative

# Create submission file
submission = pd.DataFrame({
    'ID': sample_submission['ID'],
    'audience_count': predictions
})

submission.to_csv('submission.csv', index=False)
print("\nSubmission file created: submission.csv")
print(f"Predictions range: {predictions.min():.2f} to {predictions.max():.2f}")
print(f"Mean prediction: {predictions.mean():.2f}")

print("\n" + "=" * 80)
print("PIPELINE COMPLETE!")
print("=" * 80)

In [ ]:
# submission.to_csv('/kaggle/working/submission.csv', index=False)

sam_sub = pd.read_csv('/kaggle/working/submission.csv')
sam_sub.head()

In [ ]:
sam_sub.to_csv('submission.csv',index=False)